[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/danpele/MFM/blob/main/Quantlets/Ch_00/MFM_ch0_history/MFM_ch0_history.ipynb)

# MFM_ch0_history

**Modelling Financial Markets (MFM) — Chapter 0: Financial Markets in 2026**

Two timelines for the history part of Chapter 0: exchanges and traded securities versus bubbles, panics and crashes, 1250-2026 (Bruges, Antwerp, the VOC shares of 1602, London, the Buttonwood Agreement, Bucharest 1882; tulips 1637, 1720, 1792, 1873, 1929, 1987, 2008, 2020); and milestones of financial modelling, 1900-2026, in seven lanes from random walks and portfolio theory to machine learning, generative AI and quantum computing.

*Author: Daniel Traian Pele*

In [ ]:
import os
import io
import json
import urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import warnings
warnings.filterwarnings('ignore')

## Style and data loader

In [ ]:
# Chart style: transparent background, legend below the plot
plt.rcParams['figure.facecolor'] = 'none'
plt.rcParams['axes.facecolor'] = 'none'
plt.rcParams['savefig.facecolor'] = 'none'
plt.rcParams['savefig.transparent'] = True
plt.rcParams['axes.grid'] = False
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Helvetica', 'Arial', 'DejaVu Sans']
plt.rcParams['font.size'] = 9
plt.rcParams['axes.labelsize'] = 10
plt.rcParams['axes.titlesize'] = 11
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
plt.rcParams['axes.linewidth'] = 0.6
plt.rcParams['lines.linewidth'] = 1.1
plt.rcParams['legend.facecolor'] = 'none'
plt.rcParams['legend.framealpha'] = 0
plt.rcParams['legend.fontsize'] = 8

# Chart colours
MainBlue = '#1A3A6E'
IDAred   = '#CD0000'
Forest   = '#2E7D32'
Amber    = '#B5853F'
Orange   = '#E67E22'
Purple   = '#8E44AD'
Crimson  = '#DC3545'
Gray     = '#7F7F7F'
LightGray = '#DADADA'
Teal     = '#17A2B8'


def save_fig(name):
    """Save the figure as transparent PDF and PNG in the current folder."""
    plt.savefig(f'{name}.pdf', bbox_inches='tight', transparent=True)
    plt.savefig(f'{name}.png', bbox_inches='tight', transparent=True, dpi=180)
    plt.show()
    plt.close()
    print(f"   saved {name}")


def legend_outside_bottom(ax, ncol=2, y=-0.22):
    """Place the legend outside the plot, bottom centre."""
    ax.legend(loc='upper center', bbox_to_anchor=(0.5, y), ncol=ncol, frameon=False)


def bottom_legend(fig, ncol=3, handles=None, labels=None, fontsize=8):
    """Legend below the figure (outside the axes), after tight_layout; save_fig keeps it (bbox_inches='tight')."""
    plt.tight_layout()
    if handles is None:
        handles, labels, seen = [], [], set()
        for ax in fig.axes:
            for h, l in zip(*ax.get_legend_handles_labels()):
                if l not in seen and not l.startswith('_'):
                    handles.append(h); labels.append(l); seen.add(l)
    fig.legend(handles, labels, loc='upper center', bbox_to_anchor=(0.5, 0.0), ncol=ncol, frameon=False, fontsize=fontsize)


def log_axis(ax, ticks):
    """Log axis with readable labels (no scientific notation)."""
    ax.set_yscale('log')
    ax.set_yticks(ticks)
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'{v:g}'))
    ax.yaxis.set_minor_formatter(plt.NullFormatter())


In [ ]:
REPO_RAW = 'https://raw.githubusercontent.com/danpele/MFM/main/data/market/'
LOCAL_DIRS = ['../../../data/market', '../../data/market', '../data/market', 'data/market']   # cloned repository
START, END = '2000-01-01', '2026-09-18'

# name -> (source, symbol, field)
SERIES = {
    'S&P 500': ('market', 'GSPC.INDX', 'close'),
    'Euro Stoxx 50': ('market', 'STOXX50E.INDX', 'close'),
    'Nikkei 225': ('market', 'N225.INDX', 'close'),
    'VIX': ('market', 'VIX.INDX', 'close'),
    'BET-TR': ('market', 'BETTR.INDX', 'close'),
    'BET': ('market', 'BET', 'close'),
    'BET-XT': ('market', 'BETXT.INDX', 'close'),
    'US Treasuries 20y+ (TLT)': ('market', 'TLT.US', 'adjusted_close'),
    'SPY': ('market', 'SPY.US', 'adjusted_close'),
    'RSP': ('market', 'RSP.US', 'adjusted_close'),
    'IBIT': ('market', 'IBIT.US', 'close'),
    'IBIT volume': ('market', 'IBIT.US', 'volume'),
    'Gold': ('market', 'XAUUSD.FOREX', 'close'),
    'EUR/USD': ('market', 'EURUSD.FOREX', 'close'),
    'EUR/RON (EODHD)': ('market', 'EURRON.FOREX', 'close'),
    'Bitcoin': ('market', 'BTC-USD.CC', 'close'),
    'Ethereum': ('market', 'ETH-USD.CC', 'close'),
    'Tether (USDT)': ('market', 'USDT-USD.CC', 'close'),
    'Romania 10y': ('market', 'RO10Y.GBOND', 'close'),
    'Germany 10y': ('market', 'DE10Y.GBOND', 'close'),
    'Banca Transilvania': ('market', 'TLV.RO', 'adjusted_close'),
    'OMV Petrom': ('market', 'SNP.RO', 'adjusted_close'),
    'BRD': ('market', 'BRD.RO', 'adjusted_close'),
    'Transgaz': ('market', 'TGN.RO', 'adjusted_close'),
    'Romgaz': ('market', 'SNG.RO', 'adjusted_close'),
    'Hidroelectrica': ('market', 'H2O.RO', 'adjusted_close'),
    'EUR/RON': ('bnr', 'EUR', None),
    'DGS10': ('fred', 'DGS10', None),
    'DGS2': ('fred', 'DGS2', None),
    'FEDFUNDS': ('fred', 'FEDFUNDS', None),
    'USD per EUR': ('fred', 'DEXUSEU', None),
    'JPY per USD': ('fred', 'DEXJPUS', None),
    'T10Y2Y': ('fred', 'T10Y2Y', None),
    'ETF equities': ('fred', 'BOGZ1LM563064100Q', None),
    'All equities': ('fred', 'BOGZ1LM893064105Q', None),
    'Stablecoins': ('defillama', 'stablecoincharts/all', None),
}
WEEKDAYS_ONLY = {'EUR/USD', 'Gold'}


def read_market(symbol):
    """Daily market series: local copy if the repository is cloned, otherwise GitHub."""
    for d in LOCAL_DIRS:
        path = os.path.join(d, f'{symbol}.csv')
        if os.path.exists(path):
            return pd.read_csv(path, index_col='date', parse_dates=True)
    return pd.read_csv(REPO_RAW + f'{symbol}.csv', index_col='date', parse_dates=True)


def fetch_daily(symbol, source='market', field='close', start=START, end=END):
    """Single access point to the data: returns a daily pd.Series (no NaN)."""
    if source == 'market':
        s = read_market(symbol)[field]
    elif source == 'fred':
        raw = urllib.request.urlopen(
            f'https://fred.stlouisfed.org/graph/fredgraph.csv?id={symbol}', timeout=90).read().decode()
        s = pd.read_csv(io.StringIO(raw), index_col=0, parse_dates=True, na_values='.').iloc[:, 0]
    elif source == 'defillama':
        req = urllib.request.Request(f'https://stablecoins.llama.fi/{symbol}',
                                     headers={'User-Agent': 'Mozilla/5.0'})
        d = json.load(urllib.request.urlopen(req, timeout=90))
        s = pd.Series({pd.to_datetime(int(x['date']), unit='s'): x['totalCirculatingUSD'].get('peggedUSD', 0)
                       for x in d}) / 1e9                                   # billion USD
    elif source == 'bnr':
        # official BNR rate (RON per unit of currency), yearly XML archives
        import re
        rows = []
        for y in range(max(int(start[:4]), 2005), int(end[:4]) + 1):
            url = f'https://curs.bnr.ro/files/xml/years/nbrfxrates{y}.xml'
            xml = urllib.request.urlopen(urllib.request.Request(url, headers={'User-Agent': 'Mozilla'}),
                                         timeout=90).read().decode()
            for d, body in re.findall(r'<Cube date="([\d-]+)">(.*?)</Cube>', xml, re.S):
                m = re.search(rf'<Rate currency="{symbol}">([\d.]+)</Rate>', body)
                if m:
                    rows.append((d, float(m.group(1))))
        s = pd.Series(dict(rows))
    else:
        raise ValueError(f'unknown source: {source}')
    s = pd.to_numeric(s, errors='coerce')
    s.index = pd.to_datetime(s.index)
    s.index.name = 'Date'
    return s.sort_index().loc[start:end].dropna()


def load(name, start=START, end=END):
    """The series named in SERIES (gold and EUR/USD: weekdays only)."""
    source, symbol, field = SERIES[name]
    s = fetch_daily(symbol, source, field, start, end).rename(name)
    if name in WEEKDAYS_ONLY:
        s = s[s.index.dayofweek < 5]
    return s


def load_panel(names, start=START, end=END):
    """Several series aligned on date (NaN where a market does not trade)."""
    return pd.concat([load(n, start, end) for n in names], axis=1)


## Helper functions

In [ ]:
def _timeline(ax, rows, x0, x1, levels=(0.22, -0.22, 0.40, -0.40), fs=6.3):
    """rows: list of (label, colour, [(year, text, level_index[, dx]), ...]); one horizontal lane per row."""
    for r, (lab, col, events) in enumerate(rows):
        y = -r
        ax.plot([x0, x1], [y, y], color=col, lw=0.6, alpha=0.5, zorder=1)
        ax.plot([], [], 'o', color=col, ms=5, label=lab)
        for ev in events:
            if len(ev) == 4 and isinstance(ev[1], (int, float)):   # interval (start, end, text, level)
                a, b, txt, lv = ev
                ax.plot([a, b], [y, y], color=col, lw=4, solid_capstyle='butt', zorder=2)
                xm = (a + b) / 2
            else:
                xm, txt, lv = ev[:3]
                ax.plot(xm, y, 'o', color=col, ms=4.5, zorder=3)
            dx = ev[3] if len(ev) == 4 and not isinstance(ev[1], (int, float)) else 0
            dy = levels[lv]
            ax.annotate(txt, xy=(xm, y), xytext=(xm + dx, y + dy), ha='center', va='bottom' if dy > 0 else 'top',
                        fontsize=fs, color='black', arrowprops=dict(arrowstyle='-', color=col, lw=0.5))
    ax.set_xlim(x0, x1)
    ax.set_ylim(-len(rows) + 0.4, 0.62)
    ax.set_yticks([])
    ax.spines['left'].set_visible(False)

## Data

## Charts

In [ ]:
def fig_market_history():
    """Exchanges (top lane) and bubbles/crashes (bottom lane), 1250-2026."""
    ex = [(1270, 1500, 'Bruges, Ter Beurze\n(13th-15th c.)', 0), (1531, 'Antwerp\n1531', 1), (1602, 'VOC shares\n1602', 0),
          (1698, "London,\nJonathan's\n1698", 2, -14), (1773, 'London\n"Stock Exchange"\n1773', 0),
          (1792, 'Buttonwood\n1792', 1), (1882, 'Bucharest\n1882', 0)]
    cr = [(1637, 'Tulips\n1637', 1), (1720, 'Mississippi,\nSouth Sea\n1720', 0), (1792, 'Panic\n1792', 1),
          (1873, '1873', 0), (1929, '1929', 1), (1987, '1987', 0), (2008, '2008', 1), (2020, '2020', 2)]
    rows = [('Exchanges and traded securities', MainBlue, ex), ('Bubbles, panics and crashes', IDAred, cr)]
    fig, ax = plt.subplots(figsize=(5.5, 2.2))
    _timeline(ax, rows, 1250, 2035, levels=(0.18, -0.18, 0.42, -0.42), fs=7)
    ax.set_ylim(-1.6, 0.85)
    ax.set_xticks(range(1300, 2001, 100))
    bottom_legend(fig, ncol=2)
    save_fig('ch0_market_history')

In [ ]:
def fig_model_history():
    """Milestones of financial modelling, 1900-2026, one lane per family."""
    rows = [
        ('Random walk and efficiency', MainBlue,
         [(1900, 'Bachelier', 0), (1963, 'Mandelbrot', 1, -3), (1965, 'Samuelson', 0), (1970, 'Fama', 1, 2)]),
        ('Portfolio and equilibrium', Forest,
         [(1952, 'Markowitz', 0), (1958, 'Tobin', 1), (1964, 'CAPM', 0), (1976, 'APT', 1), (1993, 'Fama-French', 0)]),
        ('Derivatives, continuous time', Purple,
         [(1973, 'Black-Scholes-Merton', 0, -5), (1977, 'Vasicek', 1), (1985, 'CIR', 0), (1992, 'HJM', 1),
          (1993, 'Heston', 2), (2018, 'rough vol.', 0)]),
        ('Volatility and risk', IDAred,
         [(1982, 'ARCH', 0, -3), (1986, 'GARCH', 1, -2), (1994, 'RiskMetrics', 0, -2), (1999, 'coherent', 1, -4),
          (2000, 'copula', 2, 2), (2003, 'realised vol.', 3, 4), (2019, 'FRTB: ES', 1, 4)]),
        ('Machine learning', Orange,
         [(1994, 'neural nets', 0), (2019, 'deep hedging', 1, -3), (2020, 'ML asset pricing', 0, 2)]),
        ('Generative AI', Amber,
         [(2017, 'Transformer', 0, -4), (2023, 'BloombergGPT', 1), (2024, 'Chronos', 0, 3)]),
        ('Quantum computing', Teal,
         [(2015, 'quantum MC', 1, -8), (2018, 'option pricing', 0), (2021, 'advantage threshold', 3, 3)]),
    ]
    fig, ax = plt.subplots(figsize=(6.2, 3.4))
    _timeline(ax, rows, 1895, 2033, levels=(0.17, -0.17, 0.34, -0.34), fs=7)
    ax.set_ylim(-len(rows) + 0.35, 0.5)
    ax.set_xticks(range(1900, 2021, 20))
    ax.tick_params(axis='x', labelsize=8)
    bottom_legend(fig, ncol=4, fontsize=7.5)
    save_fig('ch0_model_history')

## Run

In [ ]:
fig_market_history()
fig_model_history()

![ch0_market_history](./ch0_market_history.png)

![ch0_model_history](./ch0_model_history.png)

Output: `ch0_market_history.pdf`, `ch0_model_history.pdf`